# RL Systems 07 — rollout→train 的系统骨架

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Papa-Panda/field-notes/blob/main/pytorch/rl-systems/07-rl-systems.ipynb)

- **日期**：2026-10-02
- **定位**：RL track 的 RL systems coding 轮：不写新 loss，把"生成—打分—训练"三段搭成能跑的系统，接口契约和同步/异步的取舍讲清楚。
- **环境**：纯 PyTorch，CPU 单进程模拟分布式时序（看的是机制，不是吞吐）。

配套笔记：[07-rl-systems.md](07-rl-systems.md)


## 0. 因果链定位

前几篇的 loss 都默认"样本已经躺在手边"。在线 RL 的样本是 policy 自己生成的，于是多出 SFT 没有的系统问题：

- **生成（rollout）**：自回归解码，瓶颈在显存带宽，要靠推理引擎喂饱；
- **训练（learner）**：大矩阵乘，瓶颈在算力。

两个负载画像完全不同，绑在一起轮流跑就是互相等待。这一轮考的就是：怎么拆、接口存什么、同步还是异步、旧样本怎么修正。

玩具设定沿用 Research 03 的 bandit（6 个动作选目标动作 3），把结构换成 rollout/store/learner 三段。


## 1. 接口契约：Trajectory 里必须存什么

prompt、completion token、**采样时的 logp**、reward、**策略版本号** —— 后两样是异步修正和 debug 的命根子：没 logp 算不了重要性 ratio，没版本号不知道样本旧到什么程度。


In [ ]:
from dataclasses import dataclass, field
import torch

@dataclass
class Trajectory:
    prompt_id: int
    action: int                 # toy 版 completion：一个动作
    old_logp: float             # 采样时行为策略给出的 logp（off-policy 修正要用）
    reward: float
    policy_version: int         # 采样时策略的版本号（新鲜度过滤要用）

@dataclass
class TrajectoryStore:
    buf: list = field(default_factory=list)
    def add(self, trajs): self.buf.extend(trajs)
    def drain(self):
        out, self.buf = self.buf, []
        return out


## 2. 同步版：rollout 一批 → 训练 → 再 rollout

样本永远 on-policy（版本差 0），结构简单。代价是 learner 训练时 rollout 在等、rollout 时 learner 在等 —— 先把这个基线跑通。


In [ ]:
def grpo_advantages(rewards):  # 与 Research 03 同款：组内标准化
    m = rewards.mean(dim=1, keepdim=True)
    s = rewards.std(dim=1, keepdim=True)
    return (rewards - m) / (s + 1e-8)

torch.manual_seed(0)
K, TARGET, G = 6, 3, 16
logits = torch.zeros(K, requires_grad=True)
opt = torch.optim.Adam([logits], lr=0.1)
version = 0
store = TrajectoryStore()

for round_ in range(6):
    # ---- rollout：用当前策略采样 G 条，写进 store ----
    with torch.no_grad():
        lp = torch.log_softmax(logits, -1)
        actions = torch.multinomial(lp.exp(), G, replacement=True)
        for a in actions.tolist():
            store.add([Trajectory(0, a, lp[a].item(), 1.0 if a == TARGET else 0.0, version)])
    # ---- learner：消费 store，GRPO 更新，版本 +1 ----
    trajs = store.drain()
    adv = grpo_advantages(torch.tensor([t.reward for t in trajs]).view(1, -1)).view(-1)
    lp = torch.log_softmax(logits, -1)
    loss = -(torch.stack([lp[t.action] for t in trajs]) * adv).mean()
    opt.zero_grad(); loss.backward(); opt.step()
    version += 1
    print(f'round {round_}: P(目标动作) = {lp.exp()[TARGET].item():.3f}  （样本版本差恒为 0 = on-policy）')


## 3. 异步版：样本会旧，旧到什么程度要量化

改成：learner 每训一步，rollout worker 还在用 $ k $ 步前的权重生成。样本带着旧版本号进来，learner 有两个动作：

1. **算滞后** $ \text{lag} = v_{learner} - v_{sample} $ ，太旧的直接丢；
2. **重要性修正**：真实 ratio $ \rho = \exp(\log\pi_{new} - \log\pi_{old}) $ ，截断在 $[1-\epsilon, 1+\epsilon] $ 内，防旧样本把更新带偏。


In [ ]:
torch.manual_seed(0)
logits = torch.zeros(K, requires_grad=True)
opt = torch.optim.Adam([logits], lr=0.1)
snapshot = logits.detach().clone()   # rollout 用的权重快照（旧策略）
snapshot_version = 0                 # 快照对应的版本号
version = 0                          # learner 当前版本
lag_hist, kept = [], 0

for step in range(60):
    # rollout 用 snapshot 生成，trajectory 记快照的版本号
    with torch.no_grad():
        lp_old = torch.log_softmax(snapshot, -1)
        actions = torch.multinomial(lp_old.exp(), G, replacement=True)
    lp_new_all = torch.log_softmax(logits, -1)
    rewards = torch.tensor([1.0 if a == TARGET else 0.0 for a in actions.tolist()]).view(1, -1)
    adv = grpo_advantages(rewards).view(-1)

    lag = version - snapshot_version          # 这批样本旧了多少个版本
    if lag <= 5:                              # 新鲜度过滤：太旧整批丢弃
        losses = []
        for a, old_lp_i, adv_i in zip(actions.tolist(), lp_old[actions].tolist(), adv.tolist()):
            ratio = torch.exp(lp_new_all[a] - old_lp_i)
            ratio = torch.clamp(ratio, 0.8, 1.2)     # 重要性截断
            losses.append(-ratio * adv_i)
        loss = torch.stack(losses).mean()
        opt.zero_grad(); loss.backward(); opt.step()
        kept += G
    lag_hist.extend([lag] * G)
    version += 1
    if version % 20 == 0:                     # 权重推送：每 20 步给 rollout 换新权重
        snapshot = logits.detach().clone()
        snapshot_version = version

import collections
print('样本滞后分布（lag: 条数）:', dict(sorted(collections.Counter(lag_hist).items())))
print(f'保留样本比例: {kept / len(lag_hist):.2f} | 最终 P(目标动作) = {torch.log_softmax(logits, -1).exp()[TARGET].item():.3f}')


## 4. 事故清单（和 MD 里的清单对着看）

- **症状：盲训旧样本** —— 不记版本号就没法算 lag，上面的过滤和截断都无从谈起。
  **修复**：异步系统的第一个工程动作永远是给样本打版本戳（Trajectory 契约里的 `policy_version`，第 1 节）。
- **症状：logp 对不齐** —— rollout 引擎和 learner 各算一遍 logp，数值精度/实现差一点，ratio 就系统性偏移。
  **修复**：上线前做 logp 一致性对拍（同一批 token 两边 logp 差应 ≈ 0）。
- **症状：权重推送竞态** —— 生成中途换权重，同一条 trajectory 前半旧策略后半新策略。
  **修复**：按版本切分生成批次，换权重只在批次边界发生。
- **症状：reward 拖死流水线** —— reward 是外部服务，一批卡住全体等。
  **修复**：慢调用必须有超时 + 降级（如先用旧 reward 模型顶着）。

## 小结

- RL 系统的主线问题是负载错配：生成吃带宽、训练吃算力，拆开部署是默认答案。
- Trajectory 契约五件套：prompt、token、采样 logp、reward、版本号 —— 缺一个，异步就盲。
- 异步的两把锁：版本滞后过滤 + 重要性截断；同步的两笔账：利用率气泡。
- 权重推送决定滞后分布：它不是运维细节，是算法正确性的一部分。

**至此阶段 1–3 的 notebook 全部就位**（Research 01–04、Day 01 DDP、Parallel 05/06/08、RL Systems 07）。剩下的只有阶段 4 的缺口：system design、理论、behavioral —— 那些不写代码，单独准备。